In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
from mlxtend.frequent_patterns import apriori, association_rules


In [8]:
#Set display options for better readability
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.max_rows', 100)

In [9]:
# Load the cleaned dataset
df = pd.read_csv(r'E:\DA\DA\Python - Cohort Analysis\Dataset\online_retail_II_cleaned.csv')
df.sort_values(by='invoicedate', ascending=False)
df['suffix'] = df['suffix'].fillna('').replace('','None')

# 💰 Revenue Analysis

In [10]:
# KPI Metrics
total_revenue = df['total_sales'].sum()
total_orders = df['invoice'].nunique()
total_member_customers = df[df['customer_id'] != 'Guests']['customer_id'].nunique()
aov = total_revenue / total_orders

print(f"{'='*60}")
print(f"{'TOTAL REVENUE':<30} £{total_revenue:,.2f}")
print(f"{'TOTAL ORDERS':<30} {total_orders:,.0f}")
print(f"{'TOTAL MEMBER CUSTOMERS':<30} {total_member_customers:,.0f}")
print(f"{'AOV (Average Order Value)':<30} £{aov:,.2f}")
print(f"{'='*60}")

TOTAL REVENUE                  £18,859,873.22
TOTAL ORDERS                   46,881
TOTAL MEMBER CUSTOMERS         5,868
AOV (Average Order Value)      £402.29


In [31]:
#Line Chart Revenue Over Time
df['invoicedate'] = pd.to_datetime(df['invoicedate']) 
monthly_revenue = df.resample('ME', on='invoicedate')['total_sales'].sum()
fig = px.line(monthly_revenue, x=monthly_revenue.index, y=monthly_revenue.values, title='Monthly Revenue Over Time', labels={'x': 'Month', 'y': 'Total Revenue'},markers=True,template='plotly_white')
fig.update_layout(xaxis_title='Month', yaxis_title='Total Revenue')
fig.update_traces(hovertemplate='<b>Month:</b> %{x|%Y-%m}<br><b>Total Revenue:</b> £%{y:,.2f}<extra></extra>')
fig.show()

<span style="font-size: 15px;">**Trend:** Revenue is highly dependent on seasonality, especially around Black Friday (November), which can reach approximately 2.5 - 3 times higher than regular months. However, the dataset ends on 12/10/2011, so it cannot yet be concluded whether this is a trend every year.</span>

In [12]:
#Refund Over Time
refunds = df.loc[df['quantity'] < 0].copy()
refunds['invoicedate'] = pd.to_datetime(refunds['invoicedate'])
monthly_refunds = refunds.resample('ME', on='invoicedate')['total_sales'].sum().reset_index()
monthly_refunds.columns = ['month', 'total_refunds']
monthly_refunds['total_refunds'] = monthly_refunds['total_refunds'].abs()
fig = px.line(monthly_refunds, x='month', y='total_refunds', title='Monthly Refunds Over Time', labels={'x': 'Month', 'y': 'Total Refunds'}, markers=True, template='plotly_white')
fig.update_layout(xaxis_title='Month', yaxis_title='Total Refunds')
fig.update_traces(hovertemplate='<b>Month:</b> %{x|%Y-%m}<br><b>Total Refunds:</b> £%{y:,.2f}<extra></extra>')
fig.show()

<span style="font-size: 15px;">Meanwhile, the refund rate in January 2011 was high, coinciding with revenue decline from December 2011.</span>

In [13]:
#Bar Chart Top 10 Products by Revenue
df['product_id'] = df['product_id'].astype(str)
top_products = df.groupby(['product_id', 'description']).agg({'total_sales': 'sum', 'quantity': 'sum'}).sort_values('total_sales', ascending=False).head(10).reset_index()
fig = px.bar(top_products, x='total_sales', y='product_id', orientation='h', title='Top 10 Products by Total Revenue', labels={'x': 'Total Revenue', 'y': 'Product ID'}, template='plotly_white', custom_data=['description', 'quantity'])
fig.update_layout(xaxis_title='Total Revenue', yaxis_title='Product ID')
fig.update_traces(hovertemplate='<b>Product ID:</b> %{y}<br><b>Total Revenue:</b> £%{x:,.2f}<br><b>Description:</b> %{customdata[0]}<br><b>Total Quantity Sold:</b> %{customdata[1]}<extra></extra>')
fig.update_yaxes(autorange='reversed')
fig.show()

<span style="font-size: 15px;">Two products are driving the main revenue: Regency Cakestand 3 Tier and White Hanging Heart T-light Holder. Overall, top products are gifts or decorative items, which aligns well with year-end purchasing needs.</span>

In [14]:
#Table show Top 10 Product, Description, Revenue
top_products_desc = df.groupby(['product_id', 'description'])['total_sales'].sum().sort_values(ascending=False).head(10).reset_index()
print(top_products_desc)

  product_id                          description  total_sales
0      22423             Regency cakestand 3 tier    313968.52
1      85123   White hanging heart t-light holder    251582.47
2      47566                        Party bunting    147079.73
3      85099              Jumbo bag red retrospot    145946.05
4      84879        Assorted colour bird ornament    128513.24
5      22086       Paper chain kit 50's christmas    116298.59
6      79321                        Chilli lights     79905.13
7      84347  Rotating silver angels t-light hldr     70967.40
8      85099                 Jumbo bag strawberry     68489.89
9      23084                   Rabbit night light     66634.59


In [15]:
#Bar Chart Top 10 Customers by Revenue
valid_customer = df[df['customer_id'] != 'Guests']
top_customers = valid_customer.groupby('customer_id').agg({'total_sales': 'sum','quantity': 'sum'}).sort_values('total_sales', ascending=False).head(10).reset_index()
fig = px.bar(top_customers, x='total_sales', y='customer_id', orientation='h', title='Top 10 Customers by Total Revenue', labels={'x': 'Total Revenue', 'y': 'Customer ID'}, template='plotly_white', custom_data=['quantity'])
fig.update_layout(xaxis_title='Total Revenue', yaxis_title='Customer ID')
fig.update_yaxes(autorange='reversed')
fig.update_traces(hovertemplate='Customer ID: %{y}<br>Total Revenue: £%{x:,.2f}<br><b>Total Quantity Purchased:</b> %{customdata[0]}')
fig.show()

<span style="font-size: 15px;">Two super customers are carrying the company, with their spending (after refunds) being £573,008 and £523,000 respectively, vastly outperforming other positions. This shows that the business is heavily dependent on a few customer/retail business groups.</span>

In [16]:
#Top 20 Customers - Revenue Contribution %
top_20_customers = valid_customer.groupby('customer_id').agg({'total_sales': 'sum'}).sort_values('total_sales', ascending=False).head(20).reset_index()
total_revenue = valid_customer['total_sales'].sum()
top_20_revenue = top_20_customers['total_sales'].sum()
top_20_percentage = (top_20_revenue / total_revenue) * 100

print(f"Total Revenue: £{total_revenue:,.2f}")
print(f"Top 20 Customers Revenue: £{top_20_revenue:,.2f}")
print(f"Top 20 Customers Contribution: {top_20_percentage:.2f}%")
print("\n" + "="*50)
print(top_20_customers.reset_index(drop=True))

Total Revenue: £16,297,663.77
Top 20 Customers Revenue: £3,520,734.17
Top 20 Customers Contribution: 21.60%

   customer_id  total_sales
0        18102    573008.64
1        14646    523202.74
2        14156    295551.11
3        14911    257443.03
4        17450    233643.91
5        13694    190154.45
6        17511    168504.44
7        12415    143107.02
8        16684    135875.29
9        15061    124982.13
10       15311    110860.46
11       13089    109517.19
12       17949     98469.06
13       16029     97066.71
14       14298     90489.31
15       15769     84183.52
16       13798     73177.50
17       15838     72738.93
18       12931     71546.97
19       17841     67211.76


<span style="font-size: 15px;">**Top 20 customers are generating up to 20% of the business's revenue, a significantly large number.**</span>

# 👥 RFM Analysis

<span style="font-size: 15px;">**RFM Analysis is calculated based on:**
**Recency**: Number of days since the last purchase compared to a defined time point (here is the last date of the dataset)
**Frequency**: Number of times this person made a purchase (buying multiple items in one transaction counts as one purchase)
**Monetary**: Total amount spent by the customer
(Refund orders are not included)

The R, F, M indices are rated on a scale from 1 to 5, divided equally into 5 ranges from lowest to highest. Where R = 1 means this person has not purchased for a very long time, F = 1 means this person purchases very rarely, M = 1 means this person's purchases have low total value.

Customer Segment is determined by combining the three R, F, and M indices, with the highest being 555 representing Champions.</span>

In [17]:
#RFM Table Creation
df_members = df[(df['customer_id'] != 'Guests') & (df['quantity'] > 0)].copy()
df_members['invoicedate'] = pd.to_datetime(df_members['invoicedate'])
latest_date = df_members['invoicedate'].max()
rfm_table = df_members.groupby('customer_id').agg({
    'invoicedate': lambda x: (latest_date - x.max()).days,
    'quantity': 'sum',
    'total_sales': 'sum'
})
rfm_table.columns = ['Recency', 'Frequency', 'Monetary']

#RFM Segmentation
rfm_table['R_Score'] = pd.qcut(rfm_table['Recency'], 5, labels=[5, 4, 3, 2, 1])
rfm_table['F_Score'] = pd.qcut(rfm_table['Frequency'], 5, labels=[1, 2, 3, 4, 5])
rfm_table['M_Score'] = pd.qcut(rfm_table['Monetary'], 5, labels=[1, 2, 3, 4, 5])
rfm_table['RFM_Score'] = rfm_table['R_Score'].astype(str) + rfm_table['F_Score'].astype(str) + rfm_table['M_Score'].astype(str)

segs = {
    r'[1-2][1-2]': 'Hibernating',
    r'[1-2][3-4]': 'At Risk',
    r'[1-2]5': 'Can\'t Loose Them',
    r'3[1-2]': 'About To Sleep',
    r'33': 'Need Attention',
    r'[3-4][4-5]': 'Loyal Customers',
    r'41': 'Promising',
    r'51': 'New Customers',
    r'[4-5][2-3]': 'Potential Loyalists',
    r'5[4-5]': 'Champions'
}

# Create Segment column
rfm_table['Segment'] = rfm_table['R_Score'].astype(str) + rfm_table['F_Score'].astype(str)
rfm_table['Segment'] = rfm_table['Segment'].replace(segs, regex=True)
print("RFM Table created successfully")
rfm_table.head()

RFM Table created successfully


,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,Segment
customer_id,,,,,,,,
12346,325,74238,77347.01,2,5,5,255,Can't Loose Them
12347,1,2967,4921.53,5,5,5,555,Champions
12348,74,2704,1658.40,3,5,4,354,Loyal Customers
12349,18,1621,3678.69,5,4,5,545,Champions
12350,309,196,294.40,2,2,2,222,Hibernating


In [18]:
# RFM KPI Metrics
total_member_customers = len(rfm_table)
champions_count = len(rfm_table[rfm_table['Segment'] == 'Champions'])
champions_rate = (champions_count / total_member_customers) * 100
avg_monetary = rfm_table['Monetary'].mean()
avg_frequency = rfm_table['Frequency'].mean()

print(f"{'='*60}")
print(f"{'TOTAL MEMBER CUSTOMERS':<30} {total_member_customers:,.0f}")
print(f"{'CHAMPIONS RATE':<30} {champions_rate:.2f}%")
print(f"{'AVG MONETARY VALUE':<30} £{avg_monetary:,.2f}")
print(f"{'AVG FREQUENCY':<30} {avg_frequency:.2f}")
print(f"{'='*60}")

TOTAL MEMBER CUSTOMERS         5,846
CHAMPIONS RATE                 13.86%
AVG MONETARY VALUE             £2,908.43
AVG FREQUENCY                  1793.08


In [19]:
#Average Recency Analysis
avg_recency = rfm_table['Recency'].mean()
median_recency = rfm_table['Recency'].median()
min_recency = rfm_table['Recency'].min()
max_recency = rfm_table['Recency'].max()

print(f"Average Recency: {avg_recency:.2f} days")
print(f"Median Recency: {median_recency:.2f} days")
print(f"Min Recency: {min_recency:.0f} days (Most Recent Customer)")
print(f"Max Recency: {max_recency:.0f} days (Oldest Customer)")
print("\n" + "="*50)

# Average Recency by Segment
avg_recency_by_segment = rfm_table.groupby('Segment')['Recency'].mean().sort_values(ascending=False).reset_index()
avg_recency_by_segment.columns = ['Segment', 'Average Recency (Days)']
print("\nAverage Recency by Customer Segment:")
print(avg_recency_by_segment)

Average Recency: 199.13 days
Median Recency: 94.00 days
Min Recency: 0 days (Most Recent Customer)
Max Recency: 738 days (Oldest Customer)


Average Recency by Customer Segment:
               Segment  Average Recency (Days)
0          Hibernating              450.061238
1              At Risk              380.953992
2     Can't Loose Them              367.403670
3       About To Sleep              111.121447
4       Need Attention              105.826087
5      Loyal Customers               65.229765
6            Promising               36.351351
7  Potential Loyalists               24.771313
8        New Customers               10.349206
9            Champions                7.456790


<span style="font-size: 15px;">The current business has an Average Recency of up to 199.13 days, meaning more than half of customers have not returned to purchase for over 6 months. This is a quite alarming figure, especially for a business with high B2B tendencies.</span>

In [20]:
#RFM Chart - Counting Segments
segment_counts = rfm_table['Segment'].value_counts().reset_index()
segment_counts.columns = ['Segment', 'Count']
segment_counts['Percentage'] = (segment_counts['Count'] / segment_counts['Count'].sum()) * 100
fig = px.treemap(segment_counts, path=['Segment'], values='Count', title='Customer Segments Distribution', template='plotly_white', custom_data=['Percentage'])
fig.update_layout(margin=dict(t=50, l=25, r=25, b=25))
fig.update_traces(hovertemplate='<b>Segment:</b> %{label}<br><b>Number of Customers:</b> %{value}<br><b>% of Grand Total:</b> %{customdata[0]:.2f}%<extra></extra>')
fig.show()

<span style="font-size: 15px;">📊 **Customers in the Champions category only account for 13.86% of the total (excluding refund orders), similar to customers in the At Risk category. Meanwhile, the largest group of customers is in the Hibernating category, accounting for up to 1/4 of the total, meaning approximately 50% of customers have abandoned or are in a critical state.**</span>

In [21]:
#RFM Chart - Segments by Revenue
segment_revenue = rfm_table.groupby('Segment')['Monetary'].sum().reset_index()
segment_revenue['Percentage'] = (segment_revenue['Monetary'] / segment_revenue['Monetary'].sum()) * 100
fig = px.treemap(segment_revenue, path=['Segment'], values='Monetary', title='Customer Segments by Total Revenue', template='plotly_white', custom_data=['Percentage'])
fig.update_layout(margin=dict(t=50, l=25, r=25, b=25))
fig.update_traces(hovertemplate='<b>Segment:</b> %{label}<br><b>Total Revenue:</b> £%{value:,.2f}<br><b>% of Grand Total:</b> %{customdata[0]:.2f}%<extra></extra>')
fig.show()

<span style="font-size: 15px;">**80% of revenue (including Champions and Loyal Customers groups) is generated by 34% of total customers. This is quite normal when comparing to a business that primarily operates B2B.**</span>

In [22]:
#RFM Scatter Chart - Recency vs Monetary
fig = px.scatter(rfm_table, x='Recency', y='Monetary', color='Segment',log_y=False, title='Recency vs Monetary Value by Segment', template='plotly_white', hover_data=['Frequency'])
fig.update_layout(xaxis_title='Recency (Days Since Last Purchase)', yaxis_title='Monetary Value (Total Revenue)')
fig.update_traces(hovertemplate='<b>Segment:</b> %{fullData.name}<br><b>Recency:</b> %{x} days<br><b>Monetary Value:</b> £%{y:,.2f}<br><b>Frequency:</b> %{customdata[0]}<extra></extra>')
fig.show()

<span style="font-size: 15px;">**However, within the Champions group, only about 15 - 20 customers are truly supporting the entire system of the business.**</span>

In [23]:
#RFM - Frequency Distribution
rfm_table['Frequency'] = rfm_table[rfm_table['Frequency'] < 800]['Frequency']
fig = px.histogram(rfm_table, x='Frequency', nbins=70, title='Frequency Distribution of Customer Purchases', template='plotly_white')
fig.update_layout(xaxis_title='Frequency (Total Number of Purchases)', yaxis_title='Count of Customers')
fig.update_xaxes(range=[0,700])
fig.update_traces(hovertemplate='<b>Purchase Frequency Range:</b> %{x}<br><b>Number of Customers:</b> %{y}<extra></extra>')
fig.show()

<span style="font-size: 15px;">**Many customers have "skipped" for quite some time. Only about 4 - 500 customers have frequent transaction flow with the business.**</span>

# 📈 Cohort Analysis

<span style="font-size: 15px;">**Conventions:**
**Invoice Month:** The time point when the customer placed an order
**Cohort Month:** The time point when the customer made their first purchase
**Cohort Index:** The time gap between the first and subsequent purchases, value = Cohort Month - Invoice Month
**x-month Retention Rate:** Total number of customers returning after x months from start time t / total number of customers starting from time t

**Cohort Matrix:** A matrix table with rows representing the time of customer's first purchase (Ex: Jan 2010), columns representing Cohort Index, and values representing Retention Rate.</span>

In [ ]:
# Cohort Analysis - Retention Matrix Creation
df_members['invoice_month'] = df_members['invoicedate'].dt.to_period('M')
df_members['cohort_month'] = df_members.groupby('customer_id')['invoicedate'].transform('min').dt.to_period('M')
df_members['cohort_index'] = (df_members['invoice_month'] - df_members['cohort_month']).apply(lambda x: x.n)
cohort_data = df_members.groupby(['cohort_month', 'cohort_index'])['customer_id'].nunique().reset_index()
cohort_pivot = cohort_data.pivot(index='cohort_month', columns='cohort_index', values='customer_id')
cohort_size = cohort_pivot.iloc[:, 0]
retention_matrix = cohort_pivot.divide(cohort_size, axis=0)
retention_matrix = retention_matrix.fillna(0)
retention_matrix.index = retention_matrix.index.astype(str)

In [42]:
# Cohort Analysis - KPI Metrics
# Calculate retention and churn rates at 2 months and 6 months

retention_2m = retention_matrix[2].mean() if 2 in retention_matrix.columns else 0
churn_2m = (1 - retention_2m) * 100
retention_2m_pct = retention_2m * 100

retention_6m = retention_matrix[6].mean() if 6 in retention_matrix.columns else 0
churn_6m = (1 - retention_6m) * 100
retention_6m_pct = retention_6m * 100

print(f"{'='*60}")
print(f"{'RETENTION RATE (2 MONTHS)':<30} {retention_2m_pct:.2f}%")
print(f"{'CHURN RATE (2 MONTHS)':<30} {churn_2m:.2f}%")
print(f"{'RETENTION RATE (6 MONTHS)':<30} {retention_6m_pct:.2f}%")
print(f"{'CHURN RATE (6 MONTHS)':<30} {churn_6m:.2f}%")
print(f"{'='*60}")

RETENTION RATE (2 MONTHS)      20.21%
CHURN RATE (2 MONTHS)          79.79%
RETENTION RATE (6 MONTHS)      13.62%
CHURN RATE (6 MONTHS)          86.38%


In [ ]:
#Cohort Analysis - Retention Matrix Visualization 
fig = px.imshow(
    retention_matrix,
    labels=dict(x='Months Since First Purchase', y='Cohort Month', color='Retention Rate'),
    x=retention_matrix.columns,
    y=retention_matrix.index,
    title='Customer Retention Rate by Cohort (%)',
    template='plotly_white',
    color_continuous_scale='Blues', 
    text_auto='.1%'
)

fig.update_layout(xaxis_title='Months Since First Purchase', yaxis_title='Cohort Month')
fig.update_traces(hovertemplate='<b>Cohort Month:</b> %{y}<br><b>Months Since First Purchase:</b> %{x}<br><b>Retention Rate:</b> %{z:.1%}<extra></extra>')
fig.show()

<span style="font-size: 15px;">**It can be said that, except for the initial customer base (customers who existed before or at the start of the dataset timeline), the return rate of subsequent customer groups is quite low, primarily during Black Friday (November 2010). This indicates that the business is not effectively retaining its customer base.**</span>

In [33]:
#Cohort Analysis - Retention Curve
cohort_data['cohort_month'] = cohort_data['cohort_month'].astype(str)
fig = px.line(cohort_data, x='cohort_index', y='customer_id', color='cohort_month', title='Customer Retention Curve by Cohort', labels={'cohort_index': 'Months Since First Purchase', 'customer_id': 'Number of Customers'}, template='plotly_white')
fig.update_layout(xaxis_title='Months Since First Purchase', yaxis_title='Number of Customers')
fig.update_traces(hovertemplate='<b>Cohort Month:</b> %{fullData.name}<br><b>Months Since First Purchase:</b> %{x}<br><b>Number of Customers:</b> %{y}<extra></extra>')
fig.show()

<span style="font-size: 15px;">**This perspective is further supported by the fact that only customers from the beginning of the dataset are the most stable customer group.**</span>

# 💎 CLV Analysis

<span style="font-size: 15px;">**CLV Assumption:** Customer Lifespan Days is calculated as the time gap between their first and last purchase

**Predictive CLV** is only calculated for customers with lifespan days > 10 days</span>

In [38]:
#CLV Analysis - Calculation & Scatter Chart
clv_data = df_members.groupby('customer_id').agg({
    'total_sales': 'sum',
    'invoicedate': lambda x: (x.max() - x.min()).days,
    'quantity': 'sum'
}).reset_index()
clv_data.columns = ['customer_id', 'total_revenue', 'customer_lifespan_days', 'total_quantity']
clv_data = clv_data[clv_data['customer_lifespan_days'] > 10]
clv_data['CLV'] = clv_data['total_revenue']/clv_data['customer_lifespan_days'].replace(0, 1)
clv_data['Gross Predictive CLV'] = clv_data['total_revenue'] + clv_data['CLV'] * 365
clv_data['Net Predictive CLV'] = clv_data['Gross Predictive CLV'] * 0.1
fig = px.scatter(clv_data, x='customer_lifespan_days', y='Gross Predictive CLV', color='total_revenue', title='Customer Lifetime Value (CLV) Analysis', labels={'customer_lifespan_days': 'Customer Lifespan (Days)', 'CLV': 'Customer Lifetime Value'}, template='plotly_white', hover_data=['customer_id', 'total_quantity', 'total_revenue'])
fig.update_layout(xaxis_title='Customer Lifespan (Days)', yaxis_title='Customer Lifetime Value (CLV)',yaxis_type='log', coloraxis_colorbar=dict(title="Total Revenue"))
fig.update_traces(hovertemplate='<b>Customer ID:</b> %{customdata[0]}<br><b>Customer Lifespan:</b> %{x} days<br><b>Gross Predictive CLV:</b> £%{y:,.2f}<br><b>Total Revenue:</b> £%{customdata[2]:,.2f}<br><b>Total Purchases:</b> %{customdata[1]}<extra></extra>')
fig.show()

<span style="font-size: 15px;">**There are a person who have expected £860.000 in CLV in next year.**</span>

In [39]:
#CLV Analysis - Distribution Histogram
fig = px.histogram(clv_data, x='Gross Predictive CLV', nbins=50, title='Distribution of Gross Predictive CLV', template='plotly_white')
fig.update_layout(xaxis_title='Gross Predictive CLV', yaxis_title='Count of Customers')
fig.update_xaxes(range=[0,500000])
fig.update_traces(hovertemplate='<b>Gross Predictive CLV Range:</b> £%{x:,.2f}<br><b>Number of Customers:</b> %{y}<extra></extra>')
fig.show()

<span style="font-size: 15px;">**Approximately 10-15 customers are predicted to potentially generate a total value (current + future) of ≥ £180.000 within the next year. These are critical customers that must be retained at all costs.**</span>

In [40]:
#CLV Analysis - CLV by RFM Segment
rfm_clv = rfm_table.reset_index().merge(clv_data[['customer_id', 'Gross Predictive CLV']], on='customer_id', how='left')
fig = px.box(rfm_clv, x='Segment', y='Gross Predictive CLV', title='Gross Predictive CLV by RFM Segment', labels={'Segment': 'RFM Segment', 'Gross Predictive CLV': 'Gross Predictive CLV'}, template='plotly_white')
fig.update_layout(xaxis_title='RFM Segment', yaxis_title='Gross Predictive CLV')
fig.update_traces(hovertemplate='<b>RFM Segment:</b> %{x}<br><b>Gross Predictive CLV:</b> £%{y:,.2f}<extra></extra>')
fig.show()

<span style="font-size: 15px;">**Nevertheless, there are still some customers in "sleep" status, particularly in the About To Sleep group, with potential revenue generation of £20,000 - £30,000 in one year. Additionally, the "Can't Loose Them" group also deserves attention, rather than focusing only on Loyal Customers and Champions groups.**</span>

# 🛒 Market Basket Analysis

<span style="font-size: 15px;">**MBA is only calculated for products with total orders greater than 100, lift (Probability of buying Consequent item) > 1.2 and support (Total number of orders containing both products) > 2%**</span>

In [30]:
#Market Basket Analysis
items_count = df_members.groupby('description').count()['invoice']
popular_items = items_count[items_count > 100].index
df_filtered = df_members[df_members['description'].isin(popular_items)]
basket = df_filtered.groupby(['invoice', 'description'])['quantity'].sum().unstack().fillna(0).applymap(lambda x: 1 if x > 0 else 0)
frequent_itemsets = apriori(basket, min_support=0.02, use_colnames=True)
rules = association_rules(frequent_itemsets, metric='lift', min_threshold=1.2)
rules = rules.sort_values('lift', ascending=False).head(10)
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']])

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_26164\1459674591.py:5: FutureWarning:

DataFrame.applymap has been deprecated. Use DataFrame.map instead.



KeyboardInterrupt: 

<span style="font-size: 15px;">**Products with high correlation levels (Lift, Confidence) are all items of similar types. Therefore, focusing on selling Bundles (e.g., selling light combos: white + pink + yellow) would be rational and help the business achieve better revenue. At the same time, the business should continue with discount programs or use top products as gifts when purchasing other products, etc.**</span>

# 📋 Summary of Key Insights

<span style="font-size: 15px;">

• Revenue is highly seasonal, peaking around Black Friday (Nov 2010–2011), driven mainly by Gifts and Decor products, followed by a sharp drop from December to January. January also shows unusually high return activity, further impacting net performance.

• Customer revenue is highly concentrated, with a small group of users contributing a disproportionate share of total sales. Around 34% of customers generate ~80% of revenue, while the top 20 customers account for ~21.6%. The dataset also contains extreme "super-whale" customers with lifetime spend exceeding £500K.

• Customer retention is weak outside seasonal peaks, with most repeat purchases coming from early cohorts. Only a small number of customers return within 100 days, indicating low long-term engagement for newer cohorts.

• Customer value is highly uneven, with CLV heavily dependent on a small base of high-value, long-term customers, increasing revenue concentration risk.

• Product co-purchasing patterns are mostly driven by similar or variant items (e.g., same product lines with different colors, sizes, or designs), with limited evidence of strong cross-category bundling.

</span>

## 6. Implications

<span style="font-size: 15px;">

•	The business is highly seasonal and overly dependent on a small group of high-value customers. 
•	Without improving retention and diversifying the customer base, revenue will remain volatile, and long-term growth will be unsustainable once top customers churn or seasonal demand fades. 

</span>

## 7. Recommendations

<span style="font-size: 15px;">

•	Focus on acquiring and retaining new customers during and immediately after the Black Friday period, when purchase demand is at its highest.
•	Create bundle promotions for products with similar colors, sizes, or designs, as these items are frequently purchased together.
•	Continue investing in loyalty programs and personalized campaigns to strengthen relationships with high-value customers.

</span>